### Intent Parser
Constraint Schema:

In [1]:
!pip install -q transformers accelerate pydantic torch sentencepiece
from typing import List, Optional
from pydantic import BaseModel, Field, ValidationError
%pip install -q --upgrade huggingface-hub>=1.3.0
%pip install -q transformers torch accelerate
%pip install -q langchain langchain-community langchain-core langchain-text-splitters
%pip install -q faiss-cpu sentence-transformers
%pip install -q xarray netcdf4
%pip install -q numpy pandas
%pip install groq



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
zsh:1: 1.3.0 not found
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, 

In [2]:

class HardConstraints(BaseModel):
    max_time_minutes: Optional[int] = Field(
        None, description="Maximum allowed cooking time in minutes"
    )
    min_protein_grams: Optional[int] = Field(
        None, description="Minimum required grams protein in grams"
    )
    max_carb_grams: Optional[int] = Field(
        None, description="Max allowed carbs in grams"
    )
    max_fat_grams: Optional[int] = Field(
        None, description="Max allowed fat in grams"
    )
    max_sodium_mg: Optional[int] = Field(
        None, description="Maximum allowed sodium in milligrams"
    )
    vegetarian: Optional[bool] = Field(
        None, description="Whether the meal must be vegetarian"
    )
    equipment_allowed: Optional[List[str]] = Field(
        None, description="List of allowed cooking equipment"
    )
    banned_ingredients: Optional[List[str]] = Field(
        None, description="List of ingredients that are not allowed"
    )


class SoftObjectives(BaseModel):
    taste_priority: Optional[float] = Field(
        0.5, ge=0.0, le=1.0, description="Importance of taste"
    )
    health_priority: Optional[float] = Field(
        0.5, ge=0.0, le=1.0, description="Importance of health"
    )
    authenticity_priority: Optional[float] = Field(
        0.5, ge=0.0, le=1.0, description="Importance of cultural authenticity"
    )


class Preferences(BaseModel):
    ingredients_exclude: Optional[List[str]] = Field(
        None, description="List of ingredients to exclude"
    )
    ingredients_include: Optional[List[str]] = Field(
        None, description="List of ingredients to include"
    )
    spice_level: Optional[float] = Field(
        None, ge=0.0, le=1.0, description="Desired spice level"
    )
    cuisines: Optional[List[str]] = Field(
        None, description="Preferred cuisines"
    )
    diet: Optional[str] = Field(
        None, description="Diet that user is following"
    )


class Intent(BaseModel):
    hard_constraints: HardConstraints
    soft_objectives: SoftObjectives
    preferences: Preferences

Use model to extract intent fromt the prompt. The intent is parsed into the intent object created above. Any extraneous fields are pruned.

In [3]:
from groq import Groq
import json
import os

# Set your Groq API key
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY", "YOUR_GROQ_API_KEY_HERE")

client = Groq()

MODEL_NAME = "llama-3.3-70b-versatile"

SYSTEM_PROMPT = """
You are a semantic parser for a cooking assistant.

Your job is to extract constraints and preferences from user input.

Rules:
- Return ONLY valid JSON
- The top-level keys 'hard_constraints', 'soft_objectives', and 'preferences' must always be present as dictionaries.
- Do NOT add extra keys
- Use null for unspecified fields within these dictionaries
- Lists must contain strings
- Floats must be between 0 and 1 when specified
- High in protein means greater than 30g
- Low in fat or carbs means less than 20g
- Low in sodium means less than 200mg
"""

def parse_intent(user_input, schema) -> Intent:

    prompt = f"""
Schema:
{schema}

User request:
{user_input}
"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.8,
        max_tokens=200
    )

    text = response.choices[0].message.content.strip()

    try:
        # Clean the LLM output to remove markdown fences
        text = clean_llm_json(text)
        data = json.loads(text)
        intent = Intent.model_validate(data)
        # remove things where intent is null
        intent = intent.model_dump(exclude_none=True)

        return intent
    except json.JSONDecodeError:
        print("Invalid JSON:", text)
        return None

In [4]:
import re

def clean_llm_json(text: str) -> str:
    # Remove ```json ... ``` blocks
    text = re.sub(r"```json", "", text)
    text = re.sub(r"```", "", text)
    return text.strip()

schema = {
  "hard_constraints": {
    "max_time_minutes": None,
    "min_protein_grams": None,
    "max_carb_grams": None,
    "max_fat_grams": None,
    "max_sodium_mg": None,
    "vegetarian": None,
    "equipment_allowed": None,
    "banned_ingredients": None
  },
  "soft_objectives": {
    "taste_priority": None,
    "health_priority": None,
    "authenticity_priority": None
  },
  "preferences": {
    "ingredients_exclude": None,
    "ingredients_include": None,
    "spice_level": None,
    "cuisines": None,
    "diet": None
  }
}

## EDIT THIS LINE FOR A NEW QUERY
user_input = "Give me an asian style tofu recipe with rice"

intent = parse_intent(user_input, schema)
print(intent)

{'hard_constraints': {'vegetarian': True}, 'soft_objectives': {}, 'preferences': {'ingredients_include': ['tofu', 'rice'], 'cuisines': ['asian']}}


##Kitchen State

In [5]:
KITCHEN_STATE = """
{
  "kitchen_state": {

    "ingredients": [

      /* === PROTEINS === */

      {
        "name": "whole chicken bone in, french butchered",
        "quantity": 1,
        "unit": "pieces",
        "state": "raw",
        "storage": "refrigerator",
        "freshness": "fresh",
        "notes": "bone in, skin on"
      },
      {
        "name": "ground beef",
        "quantity": 500,
        "unit": "grams",
        "state": "raw",
        "storage": "refrigerator",
        "freshness": "fresh",
        "notes": "80/20"
      },
      {
        "name": "eggs",
        "quantity": 10,
        "unit": "pieces",
        "state": "whole",
        "storage": "refrigerator",
        "freshness": "good"
      },
      {
        "name": "bacon",
        "quantity": 8,
        "unit": "slices",
        "state": "raw",
        "storage": "refrigerator",
        "freshness": "good"
      },
      {
        "name": "tofu",
        "quantity": 8,
        "unit": "slices",
        "state": "raw",
        "storage": "refrigerator",
        "freshness": "good"
      },
         {
        "name": "miso",
        "quantity": 8,
        "unit": "slices",
        "state": "raw",
        "storage": "refrigerator",
        "freshness": "good"
      },
      {
        "name": "salmon fillet",
        "quantity": 2,
        "unit": "pieces",
        "state": "raw",
        "storage": "freezer",
        "freshness": "frozen"
      },
      {
        "name": "firm tofu",
        "quantity": 1,
        "unit": "block",
        "state": "raw",
        "storage": "refrigerator",
        "freshness": "good"
      },

      /* === VEGETABLES === */

      {
        "name": "spinach",
        "quantity": 3,
        "unit": "cups",
        "state": "raw",
        "storage": "refrigerator",
        "freshness": "good"
      },
      {
        "name": "mushrooms",
        "quantity": 200,
        "unit": "grams",
        "state": "raw",
        "storage": "refrigerator",
        "freshness": "good"
      },
      {
        "name": "broccoli",
        "quantity": 1,
        "unit": "head",
        "state": "raw",
        "storage": "refrigerator",
        "freshness": "fresh"
      },
      {
        "name": "bell peppers",
        "quantity": 3,
        "unit": "pieces",
        "state": "raw",
        "storage": "refrigerator"
      },
      {
        "name": "zucchini",
        "quantity": 2,
        "unit": "pieces",
        "state": "raw",
        "storage": "refrigerator"
      },
      {
        "name": "carrots",
        "quantity": 5,
        "unit": "pieces",
        "state": "raw",
        "storage": "refrigerator"
      },
      {
        "name": "yellow onion",
        "quantity": 4,
        "unit": "pieces",
        "state": "whole",
        "storage": "pantry"
      },
      {
        "name": "garlic",
        "quantity": 2,
        "unit": "bulbs",
        "state": "whole",
        "storage": "pantry"
      },
      {
        "name": "russet potatoes",
        "quantity": 6,
        "unit": "pieces",
        "state": "whole",
        "storage": "pantry"
      },

      /* === FRUITS === */

      {
        "name": "lemons",
        "quantity": 3,
        "unit": "pieces",
        "state": "whole",
        "storage": "refrigerator"
      },
      {
        "name": "apples",
        "quantity": 4,
        "unit": "pieces",
        "state": "whole",
        "storage": "refrigerator"
      },
      {
        "name": "bananas",
        "quantity": 5,
        "unit": "pieces",
        "state": "ripe",
        "storage": "counter"
      },

      /* === GRAINS & CARBS === */

      {
        "name": "rigatoni pasta",
        "quantity": 400,
        "unit": "grams",
        "state": "dry",
        "storage": "pantry"
      },
      {
        "name": "spaghetti",
        "quantity": 500,
        "unit": "grams",
        "state": "dry",
        "storage": "pantry"
      },
      {
        "name": "white rice",
        "quantity": 1,
        "unit": "kilogram",
        "state": "dry",
        "storage": "pantry"
      },
      {
        "name": "quinoa",
        "quantity": 500,
        "unit": "grams",
        "state": "dry",
        "storage": "pantry"
      },
      {
        "name": "all-purpose flour",
        "quantity": 1,
        "unit": "kilogram",
        "state": "dry",
        "storage": "pantry"
      },
      {
        "name": "bread",
        "quantity": 1,
        "unit": "loaf",
        "state": "fresh",
        "storage": "counter"
      },

      /* === DAIRY === */

      {
        "name": "milk",
        "quantity": 1,
        "unit": "liter",
        "state": "liquid",
        "storage": "refrigerator"
      },
      {
        "name": "butter",
        "quantity": 250,
        "unit": "grams",
        "state": "solid",
        "storage": "refrigerator"
      },
      {
        "name": "heavy cream",
        "quantity": 250,
        "unit": "ml",
        "state": "liquid",
        "storage": "refrigerator"
      },
      {
        "name": "parmesan cheese",
        "quantity": 150,
        "unit": "grams",
        "state": "block",
        "storage": "refrigerator"
      },
      {
        "name": "mozzarella",
        "quantity": 200,
        "unit": "grams",
        "state": "shredded",
        "storage": "refrigerator"
      },

      /* === CANNED & JARRED === */

      {
        "name": "canned diced tomatoes",
        "quantity": 2,
        "unit": "cans",
        "state": "sealed",
        "storage": "pantry"
      },
      {
        "name": "coconut milk",
        "quantity": 1,
        "unit": "can",
        "state": "sealed",
        "storage": "pantry"
      },
      {
        "name": "chickpeas",
        "quantity": 2,
        "unit": "cans",
        "state": "sealed",
        "storage": "pantry"
      },

      /* === SWEETENERS === */

      {
        "name": "granulated sugar",
        "quantity": 1,
        "unit": "kilogram",
        "state": "dry",
        "storage": "pantry"
      },
      {
        "name": "brown sugar",
        "quantity": 500,
        "unit": "grams",
        "state": "dry",
        "storage": "pantry"
      },
      {
        "name": "honey",
        "quantity": 250,
        "unit": "ml",
        "state": "liquid",
        "storage": "pantry"
      }
    ],

    /* === PANTRY STAPLES === */

    "pantry_staples": {
      "oils": ["olive oil", "neutral oil", "sesame oil"],
      "vinegars": ["red wine vinegar", "apple cider vinegar", "balsamic vinegar"],
      "seasonings": [
        "salt",
        "black pepper",
        "chili flakes",
        "paprika",
        "cumin",
        "coriander",
        "oregano",
        "thyme",
        "bay leaves",
        "cinnamon",
        "nutmeg"
      ],
      "aromatics": ["garlic", "onion", "ginger"],
      "condiments": [
        "soy sauce",
        "mustard",
        "ketchup",
        "mayonnaise",
        "hot sauce",
        "fish sauce"
      ],
      "baking": ["baking soda", "baking powder", "vanilla extract"],
      "broths": ["chicken stock cubes", "vegetable stock cubes"]
    },

    /* === COOKWARE === */

    "cookware": [
      { "name": "skillet", "size": "12-inch", "material": "stainless steel" },
      { "name": "cast iron skillet", "size": "10-inch" },
      { "name": "nonstick pan", "size": "10-inch" },
      { "name": "saucepan", "size": "medium", "material": "stainless steel" },
      { "name": "stock pot", "size": "large", "material": "aluminum" },
      { "name": "dutch oven", "size": "5-quart", "material": "enameled cast iron" },
      { "name": "sheet pan", "size": "half-sheet" },
      { "name": "casserole dish", "material": "ceramic" },
      { "name": "cutting board", "material": "wood" },
      { "name": "cutting board", "material": "plastic" },
      { "name": "chef knife", "length": "8-inch" },
      { "name": "paring knife", "length": "3-inch" },
      { "name": "bread knife", "length": "8-inch" },
      { "name": "tongs" },
      { "name": "wooden spoon" },
      { "name": "silicone spatula" },
      { "name": "whisk" },
      { "name": "ladle" },
      { "name": "colander" },
      { "name": "box grater" },
      { "name": "microplane" },
      { "name": "measuring cups" },
      { "name": "measuring spoons" },
      { "name": "mixing bowls", "quantity": 4 }
    ],

    /* === APPLIANCES === */

    "appliances": [
      { "name": "stove", "type": "gas", "burners_available": 4 },
      { "name": "oven", "type": "convection", "max_temperature_c": 260 },
      { "name": "microwave", "power_watts": 1000 },
      { "name": "blender", "type": "countertop" },
      { "name": "immersion blender" },
      { "name": "food processor" },
      { "name": "stand mixer" },
      { "name": "toaster" },
      { "name": "electric kettle" },
      { "name": "rice cooker" },
      { "name": "slow cooker" },
      { "name": "refrigerator" },
      { "name": "freezer" },
      { "name": "dishwasher" }
    ],

  }
}
"""

Using kitchen state and intent, generate candidate recipes

In [6]:
RECIPE_GENERATION_PROMPT = """
You are a culinary professor. Use the provided user intent, user query, and kitchen state to generate 3 vastly different recipes in the provided format.
Try to make the recipes as different as possible.

Rules:
1. Use EXACT ingredient names from kitchen state.
2. Output ONLY the JSON block.
3. Do not include extra fields in ingredients other than name, quantity, unit, and preparation.
4. Adhere to the rules in the user intent and query.

IMPORTANT:
All quantities must be decimal floats.

Allowed:
0.5
1.25

Not allowed:
1/2
one
a pinch
to taste

Structure:
{
  "recipes": [
    {
      "recipe_name": "string",
      "description": "string",
      "ingredients_required": [
        {
          "name": "string",
          "quantity": number,
          "unit": "string",
          "preparation": "string"
        }
      ],
      "fit_to_intent": {
        "why_it_matches": "string",
        "cuisine_alignment": "string",
        "time_estimate_minutes": number
      },
      "flavor_profile": [
        {
          "ingredients": ["string"],
          "flavor_interaction": "string"
        }
      ],
      "steps": [
        {
          "step_number": number,
          "instruction": "string"
        }
      ]
    }
  ]
}
"""

In [7]:
from typing import List, Literal, Optional
from pydantic import BaseModel, Field, ConfigDict

class FlavorCombination(BaseModel):
    ingredients: List[str] = Field(description="Exact ingredient names from kitchen state")
    flavor_interaction: str = Field(description="Brief explanation of how these flavors interact")

class Step(BaseModel):
    step_number: int
    instruction: str

class Ingredient(BaseModel):
    name: str
    quantity: float | int
    unit: str
    preparation: Optional[str]

class FitToIntent(BaseModel):
    why_it_matches: str
    cuisine_alignment: str
    time_estimate_minutes: int

class Recipe(BaseModel):
    recipe_name: str
    description: str
    ingredients_required: List[Ingredient]
    fit_to_intent: FitToIntent
    flavor_profile: List[FlavorCombination]
    steps: List[Step]

class RecipeCandidates(BaseModel):
    recipes: List[Recipe]

In [8]:
import re

def generate_recipes(intent, user_input) -> RecipeCandidates:

    prompt = f"""
User request:
{user_input}

User intent:
{intent}

Kitchen state:
{KITCHEN_STATE}

Prompt:
{RECIPE_GENERATION_PROMPT}
"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0,
        max_tokens=2000
    )

    text = response.choices[0].message.content.strip()

    try:
        # Puts
        text = clean_llm_json(text)
        data = json.loads(text)
        recipes = RecipeCandidates.model_validate(data)
        return recipes
    except json.JSONDecodeError:
        print("Invalid JSON:", text)
        return None

In [9]:
candidates = generate_recipes(intent, user_input)
print(json.dumps(candidates.model_dump(), indent=2))

{
  "recipes": [
    {
      "recipe_name": "Pan-Seared Tofu with Asian Glaze",
      "description": "A sweet and savory Asian-inspired dish featuring pan-seared tofu and a sticky glaze made with soy sauce and honey",
      "ingredients_required": [
        {
          "name": "tofu",
          "quantity": 0.5,
          "unit": "block",
          "preparation": "cut into cubes"
        },
        {
          "name": "soy sauce",
          "quantity": 0.25,
          "unit": "cup",
          "preparation": "none"
        },
        {
          "name": "honey",
          "quantity": 0.1,
          "unit": "cup",
          "preparation": "none"
        },
        {
          "name": "white rice",
          "quantity": 1.0,
          "unit": "cup",
          "preparation": "cooked"
        },
        {
          "name": "sesame oil",
          "quantity": 0.05,
          "unit": "cup",
          "preparation": "none"
        }
      ],
      "fit_to_intent": {
        "why_it_matches": "T

Based off intent and candidate recipes, we will now check that each recipe meets the constraints that we outlined. If not, we will reprompt the LLM to give us new recipes. We mainly check for the following:
- Vegetarian
- Time to make the dish
- Included ingredients
- Excluded ingredients

If any of the above things were not specified by the user, then we will skip the check

In [10]:
def recipe_ingredient_names(recipe):
    return [ing.name.lower().strip() for ing in recipe.ingredients_required]


def is_vegetarian(recipe):

    non_veg_keywords = [
        "chicken", "beef", "pork", "fish", "shrimp",
        "bacon", "lamb", "turkey", "anchovy"
    ]

    ingredients = recipe_ingredient_names(recipe)

    for ing in ingredients:
        if any(meat in ing for meat in non_veg_keywords):
            return False

    return True


def recipe_matches_intent(recipe, intent):

    # Safely get hard_constraints and preferences, handling if intent is None
    if intent is None:
        hc = {}
        pref = {}
    else:
        hc = intent.get("hard_constraints", {})
        pref = intent.get("preferences", {})

    ingredients = recipe_ingredient_names(recipe)

    # -------- time constraint --------
    max_time = hc.get("max_time_minutes")
    if max_time is not None:
        if recipe.fit_to_intent.time_estimate_minutes > max_time:
            return False

    # -------- vegetarian --------
    vegetarian = hc.get("vegetarian")
    if vegetarian is True:
        if not is_vegetarian(recipe):
            return False

    # -------- banned ingredients --------
    banned = hc.get("banned_ingredients")
    if banned:
        banned = [b.lower() for b in banned]
        if any(b in ingredients for b in banned):
            return False

    # -------- excluded ingredients --------
    excluded = pref.get("ingredients_exclude")
    if excluded:
        excluded = [e.lower() for e in excluded]
        if any(e in ingredients for e in excluded):
            return False

    # -------- required ingredients --------
    required = pref.get("ingredients_include")

    if required:
        required = [r.lower() for r in required]

        if not any(
            r in ing for r in required for ing in ingredients
        ):
            return False

    return True


def is_duplicate(recipe, existing_names):
    return recipe.recipe_name.lower().strip() in existing_names


def validate_and_fix_recipes(candidates, intent, user_input):

    existing_names = set(r.recipe_name.lower().strip() for r in candidates.recipes)

    for i, recipe in enumerate(candidates.recipes):

        if recipe_matches_intent(recipe, intent):
            continue

        print(f"Recipe '{recipe.recipe_name}' failed constraints. Regenerating...")

        attempts = 0

        while attempts < 5:

            new_candidates = generate_recipes(intent, user_input)

            if not new_candidates:
                attempts += 1
                continue

            for new_recipe in new_candidates.recipes:

                if is_duplicate(new_recipe, existing_names):
                    continue

                if recipe_matches_intent(new_recipe, intent):

                    candidates.recipes[i] = new_recipe
                    existing_names.add(new_recipe.recipe_name.lower())

                    print(f"Replaced with: {new_recipe.recipe_name}")

                    attempts = 10
                    break

            attempts += 1

    return candidates

In [11]:
candidates = validate_and_fix_recipes(
    candidates,
    intent,
    user_input
)

print(candidates.model_dump_json(indent=2))

{
  "recipes": [
    {
      "recipe_name": "Pan-Seared Tofu with Asian Glaze",
      "description": "A sweet and savory Asian-inspired dish featuring pan-seared tofu and a sticky glaze made with soy sauce and honey",
      "ingredients_required": [
        {
          "name": "tofu",
          "quantity": 0.5,
          "unit": "block",
          "preparation": "cut into cubes"
        },
        {
          "name": "soy sauce",
          "quantity": 0.25,
          "unit": "cup",
          "preparation": "none"
        },
        {
          "name": "honey",
          "quantity": 0.1,
          "unit": "cup",
          "preparation": "none"
        },
        {
          "name": "white rice",
          "quantity": 1.0,
          "unit": "cup",
          "preparation": "cooked"
        },
        {
          "name": "sesame oil",
          "quantity": 0.05,
          "unit": "cup",
          "preparation": "none"
        }
      ],
      "fit_to_intent": {
        "why_it_matches": "T

Currenlty, the recipes are basic. We will create a graph that maps ingredients to odorants, to descriptors. A traversal of this graph will help us find ingredients that will augument the recipes.

In [13]:
import json

FOOD_TO_ODORANT_PATH = "rag_docs/food_to_odorants.json"
ODORANT_TO_FOOD_PATH = "rag_docs/odorants_to_foods.json"

with open(FOOD_TO_ODORANT_PATH) as f:
    food_to_odorant = json.load(f)

with open(ODORANT_TO_FOOD_PATH) as f:
    odorant_to_food = json.load(f)

def normalize(name):
    return name.lower().strip()

# normalize names of foods
food_to_odorant = {
    normalize(k): v for k, v in food_to_odorant.items()
}

odorant_to_food = {
    normalize(k): v for k, v in odorant_to_food.items()
}

# create constant time lookup tables
food_odorants = {}
odorant_foods = {}

for food, odorants in food_to_odorant.items():

    food_odorants[food] = set()

    for entry in odorants:
        odor = normalize(entry["name"])

        food_odorants[food].add(odor)

        if odor not in odorant_foods:
            odorant_foods[odor] = set()

        odorant_foods[odor].add(food)

# compute frequency of odorants
# We will give higher weight to rare odorants
# The more common an odorant is, the less humans will taste it
odorant_frequency = {
    odor: len(foods)
    for odor, foods in odorant_foods.items()
}

# takes list of ingredients, and returns odorants
def get_dish_odorants(ingredients):

    odorants = set()

    for ing in ingredients:
        ing = normalize(ing)

        if ing in food_odorants:
            odorants |= food_odorants[ing]

    return odorants

In [14]:
from collections import defaultdict

from collections import defaultdict

def find_candidates(dish):

    def canonicalize(name):
        name = normalize(name)

        # exact match
        if name in food_odorants:
            return name

        # singular fallback (carrots -> carrot)
        if name.endswith("s") and name[:-1] in food_odorants:
            return name[:-1]

        # simple substring match (firm tofu -> tofu)
        for food in food_odorants:
            if food in name or name in food:
                return food

        return None

    dish = [canonicalize(i) for i in dish]
    dish = [i for i in dish if i is not None]

    scores = defaultdict(float)
    dish_set = set(dish)

    for ingredient in dish:

        if ingredient not in food_odorants:
            continue

        for odor in food_odorants[ingredient]:

            if odor not in odorant_foods:
                continue

            weight = 1 / odorant_frequency[odor]

            for food in odorant_foods[odor]:

                if food in dish_set:
                    continue

                scores[food] += weight

    return scores


def suggest_ingredients(dish, k=10):

    scores = find_candidates(dish)

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    return ranked[:k]

def canonicalize(name):

    name = normalize(name)

    if name in food_odorants:
        return name

    # singular fallback
    if name.endswith("s") and name[:-1] in food_odorants:
        return name[:-1]

    # try substring match
    for food in food_odorants:
        if food in name or name in food:
            return food

    return None

We now plug in the prominent combos for each recipe to generate suggestions for things to add.

In [15]:
# for each candidate recipe, make a set of the prominent ingredients
for recipe in candidates.recipes:
    prominent_ingredients = set(
        ing_name.lower().strip()
        for combo in recipe.flavor_profile
        for ing_name in combo.ingredients
    )
    ranked = suggest_ingredients(prominent_ingredients)
    print(recipe.recipe_name)
    print(prominent_ingredients)
    print(ranked)
    print()

Pan-Seared Tofu with Asian Glaze
{'soy sauce', 'honey'}
[('apple', 0.5919699857063805), ('cocoa', 0.43962507545700524), ('tomato', 0.3400004211604692), ('pear', 0.3362396455635093), ('vanilla', 0.3295034603702419), ('bartlett pear', 0.3201027381816749), ('pineapple', 0.30941412235478777), ('strawberry', 0.30307207792222235), ('white wine', 0.2864411863456715), ('rum', 0.27509899549319167)]

Tofu and Vegetable Stir-Fry
{'garlic', 'soy sauce'}
[('tea', 0.7359892730251963), ('onion', 0.6683532262585715), ('mushroom', 0.4419697569884475), ('chive', 0.40724211514746034), ('leek', 0.3974387774107389), ('shiitake', 0.3901395073377462), ('tomato', 0.22391131393000444), ('coriander', 0.21813591446473132), ('cocoa', 0.21470651419067086), ('celery', 0.2089631903387362)]

Tofu and Miso Soup
{'miso', 'garlic'}
[('tea', 0.7359892730251963), ('onion', 0.6683532262585715), ('mushroom', 0.4419697569884475), ('chive', 0.40724211514746034), ('leek', 0.3974387774107389), ('shiitake', 0.3901395073377462), 

In Create_rag_indices.ipynb, we create indices for RAG. We have two indicies, one for ingredient pairing, and one for recipes. Using this pairing RAG corpus, we come up with the best ingredients to add to each recipe from the kitchen state, and outside of the kitchen state.

In [18]:
import faiss
from sentence_transformers import SentenceTransformer


RAG_DIR = "rag_docs"


pairing_index = faiss.read_index(os.path.join(RAG_DIR, "pairing_faiss.index"))
pairing_metadata = json.load(open(os.path.join(RAG_DIR, "pairing_metadata.json")))

model = SentenceTransformer("BAAI/bge-large-en-v1.5")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 20140.16it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [19]:
recipe_index = faiss.read_index(os.path.join(RAG_DIR, "recipe_faiss.index"))
recipe_metadata = json.load(open(os.path.join(RAG_DIR, "recipe_metadata.json")))

def recipe_search(query, k=6):

    qvec = model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    scores, ids = recipe_index.search(qvec, k)

    return [recipe_metadata[i] for i in ids[0]]

def pairing_search(query, k=6):

    qvec = model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    scores, ids = pairing_index.search(qvec, k)

    return [pairing_metadata[i] for i in ids[0]]

In [20]:
#Object to store recipe additions
from pydantic import BaseModel
from typing import List


class IngredientAddition(BaseModel):
    ingredient: str
    reason: str


class RecipeIngredientSuggestions(BaseModel):
    recipe_name: str
    kitchen_state_additions: List[IngredientAddition]
    external_additions: List[IngredientAddition]

In [21]:
prompt_scaffold = """
  Return a JSON object with this structure:

  {
  "recipe_name": string,
  "kitchen_state_additions": [
    {"ingredient": string, "reason": string}
  ],
  "external_additions": [
    {"ingredient": string, "reason": string}
  ]
  }

  Rules:
  - Recommend exactly 6 ingredients from the kitchen state
  - Recommend exactly 6 ingredients NOT in the kitchen state
  - Include odorant information in reason if possible
  - Do not output anything except JSON
"""
all_recipe_suggestions = []

for recipe in candidates.recipes:

    prominent_ingredients = set(
        ing_name.lower().strip()
        for combo in recipe.flavor_profile
        for ing_name in combo.ingredients
    )

    odorant_results = pairing_search(
        f"Which odorants do these ingredients have {prominent_ingredients}"
    )

    misc_results = pairing_search(
        f"What combinations are good with these ingredients {prominent_ingredients}"
    )

    prompt = f"""
      {prompt_scaffold}

      Dish name: {recipe.recipe_name}

      Dish ingredients:
      {prominent_ingredients}

      Odorant context:
      {odorant_results}

      Additional pairing context:
      {misc_results}

      Kitchen state:
      {KITCHEN_STATE}
    """

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=800
    )

    text = response.choices[0].message.content.strip()

    try:
        # Puts
        text = clean_llm_json(text)
        data = json.loads(text)
        recipes = RecipeIngredientSuggestions.model_validate(data)
        all_recipe_suggestions.append(recipes)
    except json.JSONDecodeError:
        print("Invalid JSON:", text)



In [22]:
for r in all_recipe_suggestions:
    print(r.recipe_name)
    print("external ingredients")
    for ing in r.external_additions:
        print(ing.ingredient)
        print(ing.reason)
    print("internal ingredients")
    for ing in r.kitchen_state_additions:
        print(ing.ingredient)
        print(ing.reason)

Pan-Seared Tofu with Asian Glaze
external ingredients
rice vinegar
commonly used in Asian cuisine and has a sour flavor that balances the sweetness of the honey and the savory flavor of the soy sauce
mirin
a sweet Japanese cooking wine that complements the sweetness of the honey and the savory flavor of the soy sauce
star anise
has a sweet and licorice-like flavor that complements the sweetness of the honey and the savory flavor of the soy sauce
cucumber
commonly used in Asian cuisine and has a fresh and cooling flavor that complements the other ingredients
pickled ginger
commonly served with Asian dishes and has a sour and spicy flavor that complements the other ingredients
sesame seeds
commonly used as a garnish in Asian cuisine and have a nutty flavor that complements the other ingredients
internal ingredients
soy sauce
already in the dish and has key odorants like isobutyric acid and 2-ethyl-4-hydroxy-5-methyl-3(2h)-furanone, which contribute to its sweet and savory flavor
honey
al

Now that we have smart ingredient suggestions, we create new recipes using the ingredients from kitchen state to augment the old recipes

In [23]:
# class Recipe(BaseModel):
#     recipe_name: str
#     description: str
#     ingredients_required: List[Ingredient]
#     fit_to_intent: FitToIntent
#     flavor_profile: List[FlavorCombination]
#     steps: List[Step]

# Structure:
# {
#   "recipes": [
#     {
#       "recipe_name": "string",
#       "description": "string",
#       "ingredients_required": [
#         {
#           "name": "string",
#           "quantity": number,
#           "unit": "string",
#           "preparation": "string"
#         }
#       ],
#       "fit_to_intent": {
#         "why_it_matches": "string",
#         "cuisine_alignment": "string",
#         "time_estimate_minutes": number
#       },
#       "flavor_profile": [
#         {
#           "ingredients": ["string"],
#           "flavor_interaction": "string"
#         }
#       ],
#       "steps": [
#         {
#           "step_number": number,
#           "instruction": "string"
#         }
#       ]
#     }
#   ]
# }

class Recipe_Enhanced(BaseModel):
    recipe_name: str
    description: str
    enhancements_description: str
    ingredients_required: List[Ingredient]
    steps: List[Step]

RECIPE_PROMPT = """
Create an improved version of the following recipe.

Rules:
- Use ALL of the ingredients listed under "additions_from_kitchen".
- Keep the dish recognizable.
- Integrate the new ingredients naturally into the recipe.
- Produce a full recipe with:
  - ingredient list
  - cooking steps
- Give a couple sentences about the enhancements that you made

Return JSON in this format:

{
  "recipe_name": "string",
  description: "string",
  enhancements_description: "string",
  "ingredients_required": [
        {
          "name": "string",
          "quantity": number,
          "unit": "string",
          "preparation": "string"
        }
    ],
  "steps": [
        {
          "step_number": number,
          "instruction": "string"
        }
      ]
}
"""

improved_recipes = []

for recipe, additions in zip(candidates.recipes, all_recipe_suggestions):

    added_ingredients = [
      a.ingredient for a in additions.kitchen_state_additions
    ]

    prompt = f"""
      {RECIPE_PROMPT}

      Original recipe name:
      {recipe.recipe_name}

      Original ingredients:
      {recipe.ingredients_required}

      Ingredients to add:
      {added_ingredients}
    """

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )

    text = response.choices[0].message.content.strip()

    try:
        # Puts
        text = clean_llm_json(text)
        data = json.loads(text)
        recipes = Recipe_Enhanced.model_validate(data)
        improved_recipes.append(recipes)
    except json.JSONDecodeError:
        print("Invalid JSON:", text)

In [24]:
for recipe in improved_recipes:
    print(recipe.recipe_name)
    print(recipe.description)
    print(recipe.enhancements_description)
    print(recipe.ingredients_required)
    print(recipe.steps)

Pan-Seared Tofu with Enhanced Asian Glaze
A flavorful and aromatic dish featuring pan-seared tofu, served with a rich and savory Asian-inspired glaze, infused with the warmth of ginger and garlic, and the crunch of green onions.
This improved version of the recipe incorporates additional ingredients such as ginger, garlic, and green onions to create a more complex and nuanced flavor profile, while still maintaining the dish's recognizable Asian-inspired character. The extra soy sauce, honey, and sesame oil enhance the glaze, making it more robust and sticky.
[Ingredient(name='tofu', quantity=0.5, unit='block', preparation='cut into cubes'), Ingredient(name='soy sauce', quantity=0.5, unit='cup', preparation='none'), Ingredient(name='honey', quantity=0.2, unit='cup', preparation='none'), Ingredient(name='white rice', quantity=1.0, unit='cup', preparation='cooked'), Ingredient(name='sesame oil', quantity=0.1, unit='cup', preparation='none'), Ingredient(name='ginger', quantity=2.0, unit='t

We now score the recipes based on what would taste the best

In [25]:
# gives high score if ingredients share odorants
# def odorant_score(ingredients):

#     odorants = []

#     for ing in ingredients:
#         if ing in food_to_odorant:
#             odorants += [o["name"] for o in food_to_odorant[ing]]

#     overlap = 0

#     for i in range(len(odorants)):
#         for j in range(i+1, len(odorants)):
#             if odorants[i] == odorants[j]:
#                 overlap += 1

#     return overlap

def odorant_score(ingredients):

    odorants = []

    for ing in ingredients:
        if ing in food_to_odorant:
            odorants += [o["name"] for o in food_to_odorant[ing]]

    unique = set(odorants)

    overlap = len(odorants) - len(unique)

    return overlap / max(len(unique), 1)

# gives high score if ingredients are mentioned together as a good pairing in cooking literature
def rag_score(ingredient1, ingredient2):

    ingredient1 = ingredient1.lower()
    ingredient2 = ingredient2.lower()

    query = f"flavor pairing between {ingredient1} and {ingredient2}"
    results = recipe_search(query)

    count = 0
    for result in results:
        if 'text' in result:
            result_text_lower = result['text'].lower()
            if ingredient1 in result_text_lower and ingredient2 in result_text_lower:
                count += 1

    return count

# asks LLM to evaluate dish for taste coherence
EVAL_PROMPT = """
Score the following recipe for flavor quality.

Consider:
- balance of flavor
- culinary realism

Return a ONLY A SINGLE NUMBER between 0 and 100.
DO NOT RETURN ANY EXPLANATION
"""

def llm_score(ingredients, steps) -> float:

  prompt = f"""
  {EVAL_PROMPT}

  dish ingredients:
  {ingredients}

  dish steps:
  {steps}
  """

  response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )

  score = response.choices[0].message.content.strip()

  return int(score)/100

In [26]:
scores = []
for recipe in improved_recipes:
    total_rag_score = 0
    total_odorant_score = 0
    num_pairs = 0;
    ingredients = recipe.ingredients_required
    current_llm_score_value = llm_score(ingredients, recipe.steps)
    for i in range(len(ingredients)):
        ing1 = ingredients[i]
        for j in range(i + 1, len(ingredients)):
            ing2 = ingredients[j]
            num_pairs += 1
            total_rag_score += rag_score(ing1.name, ing2.name)
            total_odorant_score += odorant_score([ing1.name, ing2.name])
            # Now ing1 and ing2 are a unique pair of distinct ingredients
    scores.append((total_rag_score/num_pairs, total_odorant_score/num_pairs, current_llm_score_value))

print(scores)

[(0.8928571428571429, 0.021430229692320557, 0.85), (0.8222222222222222, 0.015220700152207, 0.8), (0.5833333333333334, 0.03638899082253328, 0.8)]


In [27]:
final_scores =[]
for score in scores:
    overall_score = score[0] * 30 + score[1] * 30 + score[2] * 40
    final_scores.append(overall_score)

max_score = max(final_scores)
max_index = final_scores.index(max_score)
print(f"The maximum overall score is: {max_score:.2f} (Index: {max_index})\n")

best_recipe = improved_recipes[max_index]
print(f"### Best Recipe: {best_recipe.recipe_name}\n")
print(f"**Description:** {best_recipe.description}\n")
print(f"**Enhancements:** {best_recipe.enhancements_description}\n")

print("**Ingredients Required:**")
for ing in best_recipe.ingredients_required:
    prep_info = f", Preparation: {ing.preparation}" if ing.preparation else ""
    print(f"- {ing.name.capitalize()}: {ing.quantity} {ing.unit}{prep_info}")
print("\n")

print("**Steps:**")
for step in best_recipe.steps:
    print(f"- Step {step.step_number}: {step.instruction}")
print("\n")

print("### Consider adding these external ingredients to further enhance the dish:\n")
external_additions = all_recipe_suggestions[max_index].external_additions
for addition in external_additions:
    print(f"- **{addition.ingredient.capitalize()}**: {addition.reason}")


The maximum overall score is: 61.43 (Index: 0)

### Best Recipe: Pan-Seared Tofu with Enhanced Asian Glaze

**Description:** A flavorful and aromatic dish featuring pan-seared tofu, served with a rich and savory Asian-inspired glaze, infused with the warmth of ginger and garlic, and the crunch of green onions.

**Enhancements:** This improved version of the recipe incorporates additional ingredients such as ginger, garlic, and green onions to create a more complex and nuanced flavor profile, while still maintaining the dish's recognizable Asian-inspired character. The extra soy sauce, honey, and sesame oil enhance the glaze, making it more robust and sticky.

**Ingredients Required:**
- Tofu: 0.5 block, Preparation: cut into cubes
- Soy sauce: 0.5 cup, Preparation: none
- Honey: 0.2 cup, Preparation: none
- White rice: 1.0 cup, Preparation: cooked
- Sesame oil: 0.1 cup, Preparation: none
- Ginger: 2.0 tablespoons, Preparation: grated
- Garlic: 3.0 cloves, Preparation: minced
- Green on